# 자연어 처리 실습 — 워드 임베딩 · RNN/LSTM · Seq2Seq/Attention · Transformer · 사전학습 LM

강의 4차시 내용을 **numpy만으로 직접 구현**해 보는 노트북입니다.
프레임워크(PyTorch/TensorFlow) 없이 만들기 때문에, 수식이 코드로 어떻게 바뀌는지 한 줄씩 확인할 수 있습니다.

| 파트 | 내용 |
|---|---|
| A | 원-핫 인코딩과 워드 임베딩 — 유사도, 차원의 저주 |
| B | Word2Vec Skip-gram / CBOW 학습쌍 만들기 + 미니 Skip-gram 학습 |
| C | RNN 순전파와 기울기 소실 직접 확인 |
| D | LSTM 셀 구현 — 게이트가 기억을 어떻게 조절하는가 |
| E | N-gram 언어모델 + Greedy vs Beam Search |
| F | Attention 한 스텝 계산 |
| G | Self-Attention · Multi-Head · Masking · Positional Encoding |
| H | BERT 마스킹 규칙 / GPT 자기회귀 생성 / 프롬프트 조립 |

**실행 방법**: 위에서부터 순서대로 셀을 실행하세요. 모든 셀은 numpy만 있으면 돌아갑니다.

In [ ]:
import numpy as np
import math
np.set_printoptions(precision=3, suppress=True, linewidth=120)
np.random.seed(42)
print("numpy", np.__version__, "— 준비 완료")

---
## A. 원-핫 인코딩과 워드 임베딩

### A-1. 원-핫 벡터는 왜 유사도를 못 재는가

In [ ]:
vocab = ['은행', '금융', '대출', '카페', '커피', '달린다']
word2id = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

def one_hot(w):
    v = np.zeros(V)
    v[word2id[w]] = 1.0
    return v

print("은행   :", one_hot('은행'))
print("금융   :", one_hot('금융'))
print()
# 모든 단어 쌍의 내적을 계산
M = np.stack([one_hot(w) for w in vocab])
G = M @ M.T          # 그람 행렬 = 모든 쌍의 내적
print("모든 단어 쌍의 내적 행렬:")
print(G)
print()
print("대각선(자기 자신)을 뺀 나머지가 전부 0인가? ->", np.allclose(G - np.eye(V), 0))

**결론**: 서로 다른 단어의 원-핫 벡터는 **항상 직교(내적 0)** 합니다.
"은행–금융"이 "은행–달린다"보다 가깝다는 사실을 표현할 방법이 아예 없습니다.

### A-2. 워드 임베딩에서는?

In [ ]:
# 의미가 가까운 단어끼리 가깝게 손으로 설계한 7차원 임베딩
emb = {
    '은행'  : np.array([ 0.82, 0.61, 0.34,-0.12, 0.05,-0.20, 0.11]),
    '금융'  : np.array([ 0.79, 0.66, 0.28,-0.08, 0.02,-0.17, 0.09]),
    '대출'  : np.array([ 0.71, 0.58, 0.41,-0.15, 0.10,-0.11, 0.14]),
    '카페'  : np.array([-0.10, 0.05,-0.22, 0.83, 0.60, 0.18,-0.05]),
    '커피'  : np.array([-0.14, 0.02,-0.19, 0.79, 0.66, 0.21,-0.02]),
    '달린다': np.array([ 0.03,-0.31, 0.12,-0.05,-0.10, 0.75, 0.62]),
}

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

E = np.stack([emb[w] for w in vocab])
S = np.array([[cos(emb[a], emb[b]) for b in vocab] for a in vocab])

print("코사인 유사도 행렬")
print("        " + "".join(f"{w:>9}" for w in vocab))
for i, w in enumerate(vocab):
    print(f"{w:>7} " + "".join(f"{S[i,j]:9.3f}" for j in range(V)))

In [ ]:
for a, b in [('은행','금융'), ('은행','대출'), ('카페','커피'), ('은행','카페'), ('은행','달린다')]:
    print(f"cos({a}, {b}) = {cos(emb[a], emb[b]): .3f}")

### A-3. 차원의 저주 — 차원이 커지면 무슨 일이 일어나는가

고차원에서는 **무작위로 뽑은 두 벡터가 거의 항상 직교**에 가까워집니다.
즉 "가깝다/멀다"의 구분이 무의미해집니다.

In [ ]:
for d in [2, 10, 100, 1000, 10000]:
    A = np.random.randn(400, d)
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    C = A @ A.T
    off = C[~np.eye(400, dtype=bool)]
    print(f"차원 {d:>6}: 무작위 두 벡터의 코사인 유사도 평균 {off.mean(): .4f}, "
          f"표준편차 {off.std():.4f}, |cos|>0.3 비율 {np.mean(np.abs(off)>0.3)*100:5.2f}%")

차원이 커질수록 표준편차가 $1/\sqrt{d}$ 로 줄어들어 **모든 쌍이 "거의 직교"** 로 수렴합니다.
어휘 5만 개짜리 원-핫(=5만 차원)에서는 이 현상이 극단적으로 나타납니다.

### A-4. 희소성 — 메모리를 얼마나 쓰는가

In [ ]:
for V_ in [20_000, 50_000, 500_000, 13_000_000]:
    onehot_bytes = V_ * 8          # float64 1개 단어
    emb_bytes    = 300 * 8         # 300차원 임베딩 1개 단어
    print(f"어휘 {V_:>10,}개 → 단어 하나를 원-핫으로 {onehot_bytes/1024:10.1f} KB, "
          f"300차원 임베딩으로 {emb_bytes/1024:6.2f} KB  (압축률 {onehot_bytes/emb_bytes:>8,.0f}배)")

---
## B. Word2Vec — Skip-gram과 CBOW

### B-1. 학습쌍 만들기

In [ ]:
def make_pairs(tokens, center_idx, window):
    lo, hi = max(0, center_idx - window), min(len(tokens) - 1, center_idx + window)
    ctx = [tokens[i] for i in range(lo, hi + 1) if i != center_idx]
    center = tokens[center_idx]
    skipgram = [(center, c) for c in ctx]     # 입력=중심, 정답=주변 (여러 개)
    cbow     = [(tuple(ctx), center)]          # 입력=주변 전체, 정답=중심 (한 개)
    return skipgram, cbow

sent = "어제 카페 갔었어 거기 사람 많더라".split()
sg, cb = make_pairs(sent, center_idx=2, window=2)
print("문장:", sent)
print("중심 단어:", sent[2], " 윈도우: 2\n")
print(f"Skip-gram 학습쌍 {len(sg)}개")
for a, b in sg:
    print(f"   입력 {a!r} → 정답 {b!r}")
print(f"\nCBOW 학습쌍 {len(cb)}개")
for a, b in cb:
    print(f"   입력 {a} → 정답 {b!r}")

In [ ]:
# 윈도우 크기를 키우면 학습쌍 수가 어떻게 변하는가 (문장 전체 기준)
print(f"{'window':>7} {'Skip-gram 쌍':>14} {'CBOW 쌍':>10}")
for w in range(1, 6):
    total_sg = sum(len(make_pairs(sent, i, w)[0]) for i in range(len(sent)))
    total_cb = sum(len(make_pairs(sent, i, w)[1]) for i in range(len(sent)))
    print(f"{w:>7} {total_sg:>14} {total_cb:>10}")

→ 같은 문장에서 **Skip-gram은 CBOW보다 훨씬 많은 학습 신호**를 뽑아냅니다.
이것이 "Skip-gram은 느리지만 희귀 단어에 강하고, CBOW는 빠르다"는 강의 내용의 실체입니다.

### B-2. 미니 Skip-gram 직접 학습시키기

정말로 "주변 단어를 예측"하도록 학습시키면 **의미가 비슷한 단어의 벡터가 가까워지는지** 확인합니다.

In [ ]:
# 조사를 뺀 '내용어'만으로 만든 미니 코퍼스
# (은행/금융은 대출·금리·발표 같은 같은 문맥을 공유하도록 설계)
corpus = """
은행 대출 금리 상승
은행 금리 인상 발표
은행 대출 심사 강화
은행 금리 정책 발표
금융 대출 금리 규제
금융 금리 인상 발표
금융 대출 심사 완화
금융 금리 정책 규제
카페 커피 주문 대기
카페 커피 향기 가득
카페 커피 메뉴 추천
커피 카페 주문 메뉴
공원 강아지 달리기 산책
운동장 아이 달리기 연습
공원 아이 달리기 놀이
운동장 강아지 달리기 산책
""".strip().split("\n")
corpus = [line.split() for line in corpus]

words = sorted({w for line in corpus for w in line})
w2i = {w: i for i, w in enumerate(words)}
Vv = len(words)
print(f"어휘 {Vv}개:", words)

In [ ]:
# 학습쌍 수집 (window=2)
pairs = []
for line in corpus:
    for i, c in enumerate(line):
        lo, hi = max(0, i-2), min(len(line)-1, i+2)
        for j in range(lo, hi+1):
            if j != i:
                pairs.append((w2i[c], w2i[line[j]]))
pairs = np.array(pairs)
print("학습쌍 개수:", len(pairs))

D = 12                      # 임베딩 차원
rng = np.random.default_rng(0)
W_in  = rng.normal(0, 0.3, (Vv, D))   # 중심 단어 임베딩 (우리가 쓰고 싶은 것)
W_out = rng.normal(0, 0.3, (Vv, D))   # 주변 단어 임베딩

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

lr, losses = 0.1, []
for epoch in range(300):
    rng.shuffle(pairs)
    total = 0.0
    for c, o in pairs:
        h = W_in[c]                        # 중심 단어 벡터
        scores = W_out @ h                 # 모든 단어에 대한 점수
        p = softmax(scores)
        total += -np.log(p[o] + 1e-12)
        # 역전파
        dscores = p.copy(); dscores[o] -= 1.0
        dW_out = np.outer(dscores, h)
        dh = W_out.T @ dscores
        W_out -= lr * dW_out
        W_in[c] -= lr * dh
    losses.append(total / len(pairs))
    if epoch % 50 == 0 or epoch == 299:
        print(f"epoch {epoch:>3}  평균 loss = {losses[-1]:.4f}")

**중요**: 우리가 원하는 건 예측 결과가 아니라 `W_in`(임베딩)입니다.
예측 문제는 좋은 벡터를 만들기 위한 **구실 과제(pretext task)** 일 뿐입니다.

In [ ]:
def most_similar(w, topk=3):
    v = W_in[w2i[w]]
    sims = W_in @ v / (np.linalg.norm(W_in, axis=1) * np.linalg.norm(v) + 1e-12)
    order = np.argsort(-sims)
    out = [(words[i], float(sims[i])) for i in order if words[i] != w][:topk]
    return out

for w in ['은행', '금융', '공원', '운동장', '카페']:
    sim = most_similar(w)
    print(f"{w:>5} 와 가장 가까운 단어: " + ", ".join(f"{a}({b:.3f})" for a, b in sim))

In [ ]:
# 학습 전(무작위)과 비교하면 차이가 분명해집니다
W_rand = np.random.default_rng(0).normal(0, 0.3, (Vv, D))
def cos_of(mat, a, b):
    va, vb = mat[w2i[a]], mat[w2i[b]]
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))

print(f"{'단어쌍':>14} {'학습 전':>10} {'학습 후':>10} {'변화':>10}")
for a, b in [('은행','금융'), ('카페','커피'), ('공원','운동장'), ('은행','커피'), ('은행','달리기')]:
    before, after = cos_of(W_rand, a, b), cos_of(W_in, a, b)
    print(f"{a+'-'+b:>14} {before:>10.3f} {after:>10.3f} {after-before:>+10.3f}")

### 결과 읽는 법 — 여기서 아주 중요한 오해 하나를 짚고 갑니다

| 단어쌍 | 코퍼스에서의 관계 | 학습 후 유사도 |
|---|---|---|
| 은행 – 금융 | **한 번도 같이 등장하지 않음.** 대신 둘 다 대출·금리·발표·심사와 함께 나옴 | **매우 높음** |
| 공원 – 운동장 | 같이 등장하지 않음. 둘 다 달리기·강아지·아이와 함께 나옴 | **매우 높음** |
| 카페 – 커피 | **항상 붙어서 같이 등장** | **낮음** |

**왜 이럴까요?** distributional hypothesis는 *"같이 나오는 단어끼리 비슷하다"* 가 아니라
*"**같은 이웃**을 갖는 단어끼리 비슷하다"* 입니다.
은행과 금융은 서로 만난 적이 없지만 **이웃이 똑같기 때문에** 벡터가 같은 방향으로 끌려갑니다.
반대로 카페와 커피는 서로의 이웃이라서, 학습은 "카페 벡터로 커피를 **예측**하게" 만들 뿐
두 벡터를 같은 자리에 놓지는 않습니다.

> 이 구분은 자주 헷갈리는 지점입니다. Word2Vec이 잡아내는 것은 **공기(co-occurrence)** 가 아니라
> **분포적 유사성(distributional similarity)** 입니다.
>
> 주의: 코퍼스가 16줄뿐이라 결과가 완벽하진 않습니다. 실제 Word2Vec은 수십억 단어를 씁니다.

---
## C. RNN — 순전파와 기울기 소실

### C-1. RNN 셀 구현

$$h_t = \tanh(W_{hh}h_{t-1} + W_{xh}x_t), \qquad y_t = W_{hy}h_t$$

In [ ]:
class SimpleRNN:
    def __init__(self, d_in, d_hidden, d_out, seed=0):
        rng = np.random.default_rng(seed)
        self.Wxh = rng.normal(0, 0.5, (d_hidden, d_in))
        self.Whh = rng.normal(0, 0.5, (d_hidden, d_hidden))
        self.Why = rng.normal(0, 0.5, (d_out, d_hidden))
        self.d_hidden = d_hidden

    def forward(self, xs):
        """xs: (T, d_in) -> hs: (T+1, d_hidden), ys: (T, d_out)"""
        T = len(xs)
        h = np.zeros(self.d_hidden)
        hs, ys = [h], []
        for t in range(T):
            h = np.tanh(self.Whh @ h + self.Wxh @ xs[t])   # recurrence
            hs.append(h)
            ys.append(self.Why @ h)
        return np.array(hs), np.array(ys)

rnn = SimpleRNN(d_in=4, d_hidden=5, d_out=3)
xs = np.random.randn(6, 4)          # 길이 6짜리 시퀀스
hs, ys = rnn.forward(xs)
print("입력 시퀀스 길이:", len(xs))
print("hidden state 개수:", len(hs), "(h_0 포함)")
print("h_1 =", hs[1])
print("h_6 =", hs[6])
print("y_6 =", ys[5])

### C-2. 가변 길이를 정말 처리할 수 있는가 — 같은 가중치로 길이만 바꿔보기

In [ ]:
for T in [1, 3, 10, 50]:
    xs = np.random.randn(T, 4)
    hs, ys = rnn.forward(xs)
    print(f"길이 {T:>3} 입력 → 출력 {ys.shape},  마지막 hidden state 노름 {np.linalg.norm(hs[-1]):.3f}")
print("\n→ 파라미터를 하나도 늘리지 않고 어떤 길이든 처리합니다. 이것이 가중치 공유의 진짜 이유입니다.")

### C-3. 기울기 소실을 눈으로 확인하기

$\dfrac{\partial h_T}{\partial h_1}$ 를 실제로 계산해 봅니다. 이 값이 0에 수렴하면 **먼 과거를 학습할 수 없습니다.**

In [ ]:
def grad_norm_over_time(Whh_scale, T=60, d=8, seed=1):
    rng = np.random.default_rng(seed)
    Whh = rng.normal(0, 1, (d, d))
    # 스펙트럴 반지름(최대 고유값 크기)을 원하는 값으로 맞춤
    Whh = Whh / np.max(np.abs(np.linalg.eigvals(Whh))) * Whh_scale
    h = rng.normal(0, 0.1, d)
    J = np.eye(d)          # 야코비안 누적
    norms = []
    for t in range(T):
        pre = Whh @ h
        h = np.tanh(pre)
        Dg = np.diag(1 - h**2)          # tanh 미분
        J = Dg @ Whh @ J                 # dh_t / dh_0
        norms.append(np.linalg.norm(J))
    return np.array(norms)

print(f"{'스텝':>5} {'|W|=0.7':>12} {'|W|=0.95':>12} {'|W|=1.0':>12} {'|W|=1.2':>12}")
g07, g095, g10, g12 = (grad_norm_over_time(s) for s in [0.7, 0.95, 1.0, 1.2])
for t in [0, 4, 9, 19, 39, 59]:
    print(f"{t+1:>5} {g07[t]:>12.3e} {g095[t]:>12.3e} {g10[t]:>12.3e} {g12[t]:>12.3e}")

In [ ]:
print("60스텝 뒤 기울기 크기가 1스텝 때의 몇 배인가:")
for name, g in [('|W|=0.7', g07), ('|W|=0.95', g095), ('|W|=1.0', g10), ('|W|=1.2', g12)]:
    ratio = g[59] / g[0]
    if   ratio < 1e-4: verdict = "심각한 소실 — 이만큼 먼 과거는 학습 불가"
    elif ratio < 0.3:  verdict = "완만한 소실 — 문장이 더 길어지면 결국 사라짐"
    elif ratio > 1e2:  verdict = "폭발 — 값이 발산해 학습이 망가짐"
    else:              verdict = "유지 — 하지만 이 상태를 유지하기가 매우 어려움"
    print(f"  {name:>9}: {ratio:>10.3e}   → {verdict}")
print()
print("※ |W|=0.95 처럼 1에 가까워도, 길이가 200이면 0.95^200 ≈ {:.2e} 로 결국 사라집니다."
      .format(0.95**200))

**결론**: $|W|$ 가 1보다 조금만 작아도 기울기가 지수적으로 0에 수렴합니다.
그래서 RNN은 **단기 의존성만 학습**하게 됩니다. 이것을 해결한 것이 LSTM입니다.

---
## D. LSTM — 게이트로 기억을 조절하기

$$f_t=\sigma(W_f[h_{t-1},x_t]+b_f),\quad i_t=\sigma(W_i[\cdot]+b_i),\quad \tilde{C}_t=\tanh(W_C[\cdot]+b_C)$$
$$C_t=f_t*C_{t-1}+i_t*\tilde{C}_t,\qquad o_t=\sigma(W_o[\cdot]+b_o),\qquad h_t=o_t*\tanh(C_t)$$

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60, 60)))

class LSTMCell:
    def __init__(self, d_in, d_hidden, seed=0, forget_bias=0.0):
        rng = np.random.default_rng(seed)
        n = d_in + d_hidden
        s = 1.0 / np.sqrt(n)
        self.Wf = rng.normal(0, s, (d_hidden, n)); self.bf = np.full(d_hidden, forget_bias)
        self.Wi = rng.normal(0, s, (d_hidden, n)); self.bi = np.zeros(d_hidden)
        self.Wc = rng.normal(0, s, (d_hidden, n)); self.bc = np.zeros(d_hidden)
        self.Wo = rng.normal(0, s, (d_hidden, n)); self.bo = np.zeros(d_hidden)
        self.d_hidden = d_hidden

    def step(self, x, h, C):
        z = np.concatenate([h, x])                 # [h_{t-1}, x_t]
        f = sigmoid(self.Wf @ z + self.bf)         # forget gate
        i = sigmoid(self.Wi @ z + self.bi)         # input gate
        Ct = np.tanh(self.Wc @ z + self.bc)        # new cell content
        C = f * C + i * Ct                         # cell state 갱신
        o = sigmoid(self.Wo @ z + self.bo)         # output gate
        h = o * np.tanh(C)                         # hidden state
        return h, C, dict(f=f, i=i, o=o, C_tilde=Ct)

cell = LSTMCell(d_in=3, d_hidden=4, seed=3)
h = np.zeros(4); C = np.zeros(4)
xs = np.random.randn(5, 3)
print(f"{'t':>2} {'forget 평균':>12} {'input 평균':>12} {'output 평균':>12} {'|C_t|':>8} {'|h_t|':>8}")
for t, x in enumerate(xs):
    h, C, g = cell.step(x, h, C)
    print(f"{t+1:>2} {g['f'].mean():>12.3f} {g['i'].mean():>12.3f} {g['o'].mean():>12.3f} "
          f"{np.linalg.norm(C):>8.3f} {np.linalg.norm(h):>8.3f}")

### D-1. forget gate 값이 "기억의 반감기"를 결정한다

게이트를 고정값으로 두고, **첫 스텝에 넣은 정보가 몇 스텝 뒤까지 남는지** 봅니다.

In [ ]:
def memory_decay(f, T=40):
    """첫 스텝에 1을 기록한 뒤, 이후엔 새 정보를 넣지 않을 때 C_t의 변화"""
    C, trace = 1.0, []
    for t in range(T):
        trace.append(C)
        C = f * C          # i=0 이므로 C_t = f * C_{t-1}
    return np.array(trace)

print(f"{'forget gate':>12} {'10스텝 뒤':>12} {'20스텝 뒤':>12} {'40스텝 뒤':>12} {'반감기(스텝)':>14}")
for f in [0.3, 0.5, 0.7, 0.9, 0.95, 0.99]:
    tr = memory_decay(f)
    half = int(np.argmax(tr < 0.5)) if np.any(tr < 0.5) else 40
    print(f"{f:>12.2f} {tr[10]:>12.4f} {tr[20]:>12.4f} {tr[39]:>12.4f} {half:>14}")

**핵심질문 5 확인**: 학습 초기에는 시그모이드 출력이 0.5 근처라 `0.5**20 ≈ 1e-6`,
즉 20스텝만 지나도 기억이 사라집니다. 그래서 **forget gate의 bias를 크게 초기화**하는 요령을 씁니다.

In [ ]:
for fb in [0.0, 1.0, 2.0]:
    c = LSTMCell(3, 8, seed=5, forget_bias=fb)
    h = np.zeros(8); C = np.zeros(8)
    fs = []
    for x in np.random.randn(10, 3):
        h, C, g = c.step(x, h, C)
        fs.append(g['f'].mean())
    f_mean = float(np.mean(fs))
    print(f"forget bias = {fb:>3.1f} → forget gate 평균 {f_mean:.3f}, "
          f"20스텝 뒤 기억 잔존율 {f_mean**20:.2e}")

### D-2. LSTM은 왜 기울기가 덜 소실되는가

RNN의 기울기는 $\prod W$ (곱셈)인 반면, LSTM의 cell state 경로는
$C_t = f_t \cdot C_{t-1} + (\ldots)$ 이라 $\partial C_t / \partial C_{t-1} = f_t$ 로 **덧셈 구조의 지름길**이 생깁니다.

In [ ]:
T = 50
# RNN: tanh 미분(<1)까지 곱해짐
rnn_grad = np.prod([0.9 * 0.6] * T)     # W=0.9, tanh'≈0.6 가정
# LSTM: cell state 경로는 forget gate만 곱해짐 (tanh 미분이 끼어들지 않음)
lstm_grad = np.prod([0.95] * T)
print(f"{T}스텝 역전파 후 기울기 배율")
print(f"  RNN  (W·tanh' ≈ 0.54씩 곱)      : {rnn_grad:.3e}")
print(f"  LSTM (forget gate 0.95씩 곱)    : {lstm_grad:.3e}")
print(f"  → LSTM 쪽이 {lstm_grad/rnn_grad:.3e} 배 큽니다.")

---
## E. 언어모델 — N-gram, Greedy, Beam Search

### E-1. N-gram 언어모델 만들기

In [ ]:
from collections import defaultdict, Counter

corpus_text = """SSAFY 학생들은 알고리즘 을 공부 한다
SSAFY 학생들은 인공지능 을 공부 한다
SSAFY 학생들은 인공지능 을 공부 한다
SSAFY 학생들은 파이썬 을 공부 한다
SSAFY 학생들은 인공지능 을 배운다
우리 는 인공지능 을 공부 한다
학생들은 인공지능 을 공부 한다
SSAFY 학생들은 알고리즘 을 배운다
SSAFY 학생들은 인공지능 을 공부 한다
나 는 자연어 처리 를 공부 한다"""
tokens = corpus_text.split()
print("총 토큰 수:", len(tokens), " 어휘 크기:", len(set(tokens)))

def build_ngram(tokens, n):
    ctx_count = Counter()
    full_count = Counter()
    for i in range(len(tokens) - n + 1):
        ctx = tuple(tokens[i:i+n-1])
        ctx_count[ctx] += 1
        full_count[tuple(tokens[i:i+n])] += 1
    return ctx_count, full_count

def next_word_probs(tokens, n, context):
    ctx_count, full_count = build_ngram(tokens, n)
    ctx = tuple(context[-(n-1):]) if n > 1 else ()
    denom = ctx_count[ctx]
    if denom == 0:
        return {}, 0
    probs = {k[-1]: v / denom for k, v in full_count.items() if k[:-1] == ctx}
    return dict(sorted(probs.items(), key=lambda kv: -kv[1])), denom

probs, denom = next_word_probs(tokens, 4, ['SSAFY', '학생들은', '인공지능'])
print("\n4-gram: P(w | SSAFY 학생들은 인공지능),  분모 count =", denom)
for w, p in probs.items():
    print(f"   P({w:>5} | ...) = {p:.3f}")

### E-2. $n$을 키우면 왜 안 되는가 — 희소성 직접 측정

In [ ]:
V_corpus = len(set(tokens))
print(f"{'n':>3} {'실제 등장 n-gram':>18} {'단 1번만 등장':>14} {'이론상 가능 조합':>18} {'커버율':>12}")
for n in range(1, 7):
    _, full = build_ngram(tokens, n)
    observed = len(full)
    once = sum(1 for v in full.values() if v == 1)
    possible = V_corpus ** n
    print(f"{n:>3} {observed:>18,} {once:>10,} ({100*once/observed:4.0f}%) {possible:>18,.0f} "
          f"{100*observed/possible:>11.6f}%")

In [ ]:
# 학습 코퍼스에 없던 문맥이 나오면 어떻게 되는가
for ctx in [['SSAFY','학생들은','인공지능'], ['우리','는','자연어'], ['나','는','알고리즘']]:
    p, d = next_word_probs(tokens, 4, ctx)
    status = f"{len(p)}개 후보, 분모 {d}" if d > 0 else "분모가 0 → 확률 계산 불가 (0으로 나눔)"
    print(f"문맥 {ctx} → {status}")

> $n$을 키우면 예측은 정교해지지만 **count가 0인 문맥이 폭증**해서 확률을 계산할 수 없게 됩니다.
> (실무에서는 smoothing/backoff로 완화하지만 근본 해결은 아닙니다.)
> 이 한계 때문에 **세는 대신 학습해서 일반화하는** 신경망 언어모델이 등장했습니다.

### E-3. Greedy vs Beam Search

In [ ]:
# 작은 확률 트리 (첫 스텝에서 chien이 더 높지만, 그 뒤가 나쁨)
tree = {
    ():                        [('le',0.45), ('chien',0.55)],
    ('le',):                   [('chat',0.70), ('noir',0.30)],
    ('chien',):                [('noir',0.40), ('a',0.35), ('chat',0.25)],
    ('le','chat'):             [('noir',0.90), ('a',0.10)],
    ('le','noir'):             [('a',0.60), ('chat',0.40)],
    ('chien','noir'):          [('a',0.50), ('bu',0.50)],
    ('chien','a'):             [('bu',0.60), ('noir',0.40)],
    ('chien','chat'):          [('noir',0.55), ('a',0.45)],
    ('le','chat','noir'):      [('a',0.95), ('bu',0.05)],
    ('le','chat','a'):         [('bu',0.80), ('noir',0.20)],
    ('le','noir','a'):         [('bu',0.70), ('chat',0.30)],
}
EOS = '<EOS>'
def children(seq):
    return tree.get(tuple(seq), [(EOS, 1.0)])

def beam_search(k, max_len=5, verbose=True):
    beams = [([], 0.0, False)]                        # (시퀀스, 로그확률, 종료여부)
    for step in range(max_len):
        cand = []
        for seq, lp, done in beams:
            if done:
                cand.append((seq, lp, True)); continue
            for w, p in children(seq):
                cand.append((seq + [w], lp + math.log(p), w == EOS))
        cand.sort(key=lambda x: -x[1])
        beams = cand[:k]
        if verbose:
            print(f"  스텝 {step+1}: 후보 {len(cand)}개 중 상위 {k}개 유지")
            for seq, lp, _ in beams:
                print(f"     {' '.join(seq):<28} logP={lp:7.3f}  P={math.exp(lp):.4f}")
        if all(d for _, _, d in beams):
            break
    return beams[0]

print("=== Greedy (k=1) ===")
g = beam_search(1)
print(f"\n선택: {' '.join(g[0])}  전체 확률 {math.exp(g[1]):.4f}\n")
print("=== Beam Search (k=3) ===")
b = beam_search(3)
print(f"\n선택: {' '.join(b[0])}  전체 확률 {math.exp(b[1]):.4f}")

In [ ]:
print(f"{'k':>3} {'최종 문장':>28} {'전체 확률':>12}")
for k in [1, 2, 3, 4]:
    seq, lp, _ = beam_search(k, verbose=False)
    print(f"{k:>3} {' '.join(seq):>28} {math.exp(lp):>12.4f}")
print("\n→ 매 스텝 최선(greedy)의 나열과, 문장 전체 확률의 최선은 서로 다릅니다.")

---
## F. Attention 한 스텝 계산

$$e_i = \text{score}(s, h_i), \qquad \alpha = \text{softmax}(e), \qquad a = \sum_i \alpha_i h_i$$

In [ ]:
src_words = ['the', 'black', 'cat', 'drank', 'milk']
d = 6
rng = np.random.default_rng(7)
H = rng.normal(0, 1, (len(src_words), d))       # 인코더 hidden states (Values)
s = rng.normal(0, 1, d)                          # 디코더 hidden state (Query)

def attention(s, H, scale=False, verbose=True):
    e = H @ s                                    # ① dot-product score
    if scale:
        e = e / np.sqrt(H.shape[1])
    a = np.exp(e - e.max()); a = a / a.sum()     # ② softmax
    ctx = a @ H                                  # ③ 가중합
    if verbose:
        print(f"{'단어':>8} {'점수 e_i':>10} {'가중치 α_i':>12}")
        for w, ei, ai in zip(src_words, e, a):
            bar = '█' * int(ai * 40)
            print(f"{w:>8} {ei:>10.3f} {ai:>12.3f}  {bar}")
        print(f"\ncontext vector a = {ctx}")
        print(f"가중치 합 = {a.sum():.6f}  (항상 1)")
    return a, ctx

a1, c1 = attention(s, H)

### F-1. 세 가지 score 함수 비교

In [ ]:
W = rng.normal(0, 0.5, (d, d))
v = rng.normal(0, 0.5, d)
Wh = rng.normal(0, 0.5, (d, d)); Ws = rng.normal(0, 0.5, (d, d))

def score_dot(s, H):          return H @ s
def score_mult(s, H):         return H @ W @ s
def score_add(s, H):          return np.tanh(H @ Wh.T + (Ws @ s)) @ v

for name, fn in [('Dot product', score_dot), ('Multiplicative', score_mult), ('Additive', score_add)]:
    e = fn(s, H)
    a = np.exp(e - e.max()); a /= a.sum()
    print(f"{name:>16}: α = {np.round(a,3)}  가장 주목한 단어 = {src_words[int(np.argmax(a))]}")

### F-2. 스케일링이 왜 필요한가 — 차원이 커지면 softmax가 뾰족해진다

In [ ]:
print(f"{'차원 d':>8} {'점수 표준편차':>14} {'최대 α (스케일 X)':>20} {'최대 α (÷√d)':>16}")
for dd in [4, 16, 64, 256, 1024]:
    Hd = np.random.default_rng(1).normal(0, 1, (5, dd))
    sd = np.random.default_rng(2).normal(0, 1, dd)
    e = Hd @ sd
    a_raw = np.exp(e - e.max()); a_raw /= a_raw.sum()
    e2 = e / np.sqrt(dd)
    a_sc = np.exp(e2 - e2.max()); a_sc /= a_sc.sum()
    print(f"{dd:>8} {e.std():>14.2f} {a_raw.max():>20.4f} {a_sc.max():>16.4f}")

> 차원이 커질수록 내적의 분산이 $d$에 비례해 커져 softmax가 **거의 one-hot** 이 됩니다.
> 그러면 기울기가 거의 0이 되어(saturated softmax) 학습이 안 됩니다.
> $\sqrt{d}$ 로 나누면 분산이 1 부근으로 고정되어 분포가 안정됩니다. — 강의의 Scaled Dot Product

---
## G. Self-Attention과 Transformer 구성요소

### G-1. Self-Attention 한 층 구현

In [ ]:
sent = ['어제', '카페', '갔었어', '거기', '사람', '많더라']
n, d_model = len(sent), 8
rng = np.random.default_rng(11)
X = rng.normal(0, 1, (n, d_model))               # 단어 임베딩
Wq = rng.normal(0, 0.5, (d_model, d_model))
Wk = rng.normal(0, 0.5, (d_model, d_model))
Wv = rng.normal(0, 0.5, (d_model, d_model))

def softmax_rows(M):
    M = M - M.max(axis=-1, keepdims=True)
    E = np.exp(M)
    return E / E.sum(axis=-1, keepdims=True)

def self_attention(X, Wq, Wk, Wv, mask=False):
    Q, K, Vv = X @ Wq, X @ Wk, X @ Wv
    scores = Q @ K.T / np.sqrt(Wq.shape[1])       # scaled dot product
    if mask:
        m = np.triu(np.ones_like(scores), k=1).astype(bool)
        scores = np.where(m, -np.inf, scores)     # 미래를 -inf로
    A = softmax_rows(scores)
    return A @ Vv, A

out, A = self_attention(X, Wq, Wk, Wv)
print("attention 행렬 α (행=쿼리 단어, 열=참조 대상)")
print("          " + "".join(f"{w:>9}" for w in sent))
for i, w in enumerate(sent):
    print(f"{w:>9} " + "".join(f"{A[i,j]:9.3f}" for j in range(n)))
print("\n각 행의 합 =", np.round(A.sum(axis=1), 6))
print("출력 shape:", out.shape, "(입력과 동일 — 그래서 층을 계속 쌓을 수 있습니다)")

### G-2. Masked Self-Attention — 미래를 가리면

In [ ]:
out_m, A_m = self_attention(X, Wq, Wk, Wv, mask=True)
print("Masked attention 행렬")
print("          " + "".join(f"{w:>9}" for w in sent))
for i, w in enumerate(sent):
    print(f"{w:>9} " + "".join(f"{A_m[i,j]:9.3f}" for j in range(n)))
print()
upper = A_m[np.triu(np.ones_like(A_m), k=1).astype(bool)]
print("상삼각(미래) 영역의 가중치 합 =", upper.sum(), "→ 정확히 0")
print("각 행의 합 =", np.round(A_m.sum(axis=1), 6), "→ 여전히 1 (남은 위치로 재분배됨)")

In [ ]:
# -inf 대신 유한한 값을 쓰면?
def masked_with(value):
    Q, K, Vv = X @ Wq, X @ Wk, X @ Wv
    scores = Q @ K.T / np.sqrt(d_model)
    m = np.triu(np.ones_like(scores), k=1).astype(bool)
    scores = np.where(m, value, scores)
    A = softmax_rows(scores)
    return A[m].sum()

for val in [-5, -10, -1e4, -1e9, -np.inf]:
    print(f"마스크 값 {str(val):>8} → 미래로 새어 나간 확률 합 = {masked_with(val):.3e}")

### G-3. Multi-Head Attention — 차원을 쪼개 쓰는 트레이드오프

In [ ]:
def multi_head(X, n_heads, d_model=8, seed=3, mask=False):
    assert d_model % n_heads == 0
    d_head = d_model // n_heads
    rng = np.random.default_rng(seed)
    outs, As = [], []
    for h in range(n_heads):
        Wq = rng.normal(0, 0.5, (d_model, d_head))
        Wk = rng.normal(0, 0.5, (d_model, d_head))
        Wv = rng.normal(0, 0.5, (d_model, d_head))
        o, A = self_attention(X, Wq, Wk, Wv, mask=mask)
        outs.append(o); As.append(A)
    return np.concatenate(outs, axis=1), As      # concat 후 원래 차원 복구

for hcount in [1, 2, 4, 8]:
    o, As = multi_head(X, hcount)
    d_head = 8 // hcount
    # 헤드들이 얼마나 서로 다른 패턴을 보는가 = attention 행렬 간 평균 차이
    if hcount > 1:
        diffs = [np.abs(As[i] - As[j]).mean() for i in range(hcount) for j in range(i+1, hcount)]
        div = np.mean(diffs)
    else:
        div = 0.0
    print(f"head {hcount}개 → head당 차원 {d_head}, 출력 shape {o.shape}, 헤드 간 패턴 차이 {div:.4f}")

**핵심질문 11 확인**: `d_model`이 고정되어 있으므로 head를 늘리면 head당 차원이 반비례로 줄어듭니다.
head 8개면 head당 1차원 — 이 정도면 표현이 너무 거칠어집니다. 그래서 최적 head 수가 따로 존재합니다.

### G-4. Positional Encoding

In [ ]:
def positional_encoding(max_len, d):
    pos = np.arange(max_len)[:, None]
    i = np.arange(d)[None, :]
    angle = pos / np.power(10000, (2 * (i // 2)) / d)
    pe = np.where(i % 2 == 0, np.sin(angle), np.cos(angle))
    return pe

PE = positional_encoding(50, 32)
print("PE shape:", PE.shape)
print("\n위치 간 코사인 유사도 (위치 10 기준)")
for p in [10, 11, 12, 15, 20, 30, 45]:
    print(f"  위치 10 ↔ 위치 {p:>2}: {cos(PE[10], PE[p]): .4f}   (거리 {abs(p-10)})")

In [ ]:
# "단어 순서를 바꾸면 각 단어의 출력이 달라지는가?" 를 검사합니다.
perm = [3, 1, 0, 5, 2, 4]          # 단어를 섞는 순서
inv  = np.argsort(perm)             # 원래 자리로 되돌리는 순서

# ① Positional Encoding 없이
out_a, _ = self_attention(X, Wq, Wk, Wv)              # 원래 순서
out_b, _ = self_attention(X[perm], Wq, Wk, Wv)        # 섞은 순서
same_no_pe = np.allclose(out_a, out_b[inv])           # 되돌려서 비교
print("[PE 없음] 단어 순서를 바꿔도 각 단어의 출력이 같은가? ->", same_no_pe)
print("   → True. self-attention 자체는 순서를 전혀 모릅니다 (순열 등변, permutation-equivariant).")
print(f"   두 결과의 최대 차이 = {np.abs(out_a - out_b[inv]).max():.2e}\n")

# ② Positional Encoding 추가 (위치 벡터는 '자리'에 붙어 있으므로 섞지 않습니다)
P = PE[:len(X), :d_model]
out_c, _ = self_attention(X + P, Wq, Wk, Wv)          # 원래 순서 + 위치
out_d, _ = self_attention(X[perm] + P, Wq, Wk, Wv)    # 섞은 순서 + 같은 자리의 위치
same_pe = np.allclose(out_c, out_d[inv])
print("[PE 있음] 같은 검사 ->", same_pe)
print("   → False. 같은 단어라도 '몇 번째 자리에 있는가'에 따라 출력이 달라집니다.")
print(f"   두 결과의 최대 차이 = {np.abs(out_c - out_d[inv]).max():.4f}")

### G-5. Residual + LayerNorm + FFN = Transformer 블록 하나

In [ ]:
def layer_norm(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    sd = x.std(axis=-1, keepdims=True)
    return (x - mu) / (sd + eps)

def ffn(x, W1, b1, W2, b2):
    return np.maximum(0, x @ W1 + b1) @ W2 + b2      # ReLU

rng = np.random.default_rng(21)
W1 = rng.normal(0, 0.3, (d_model, d_model*4)); b1 = np.zeros(d_model*4)
W2 = rng.normal(0, 0.3, (d_model*4, d_model)); b2 = np.zeros(d_model)

def transformer_block(X, mask=False):
    attn, A = self_attention(X, Wq, Wk, Wv, mask=mask)
    X1 = layer_norm(X + attn)                        # Add & Norm
    f = ffn(X1, W1, b1, W2, b2)
    X2 = layer_norm(X1 + f)                          # Add & Norm
    return X2

Y = X.copy()
print(f"{'층':>4} {'출력 노름 평균':>16} {'층별 표준편차':>16}")
for layer in range(6):
    Y = transformer_block(Y)
    print(f"{layer+1:>4} {np.linalg.norm(Y, axis=1).mean():>16.4f} {Y.std():>16.4f}")
print("\n→ LayerNorm 덕분에 층을 깊게 쌓아도 값의 크기가 폭발/소멸하지 않습니다.")

In [ ]:
# residual이 없으면 '원래 입력 정보'가 얼마나 남는가?
def block_no_residual(X, mask=False):
    attn, _ = self_attention(X, Wq, Wk, Wv, mask=mask)
    return layer_norm(ffn(layer_norm(attn), W1, b1, W2, b2))

def keep_ratio(Y, X0):
    """각 단어 벡터가 원래 입력과 얼마나 닮았는지 (코사인 유사도 평균)"""
    return float(np.mean([cos(Y[i], X0[i]) for i in range(len(X0))]))

def spread(Y):
    """단어들끼리 서로 얼마나 구별되는지 (서로 다른 단어 쌍의 평균 거리)"""
    n = len(Y)
    return float(np.mean([np.linalg.norm(Y[i]-Y[j]) for i in range(n) for j in range(i+1, n)]))

Ya, Yb = X.copy(), X.copy()
print(f"{'층':>4} | {'residual 있음':^26} | {'residual 없음':^26}")
print(f"{'':>4} | {'원본과 유사도':>13}{'단어간 구별도':>13} | {'원본과 유사도':>13}{'단어간 구별도':>13}")
print("-" * 70)
for layer in range(6):
    Ya = transformer_block(Ya)
    Yb = block_no_residual(Yb)
    print(f"{layer+1:>4} | {keep_ratio(Ya,X):>13.3f}{spread(Ya):>13.3f} | "
          f"{keep_ratio(Yb,X):>13.3f}{spread(Yb):>13.3f}")
print()
print("→ '단어간 구별도' 열을 보세요. residual이 없으면 4개 층 만에 0으로 붕괴합니다.")
print("   즉 모든 단어가 똑같은 벡터가 되어 버려서, 층을 더 쌓아도 아무 정보가 없습니다.")
print("   (attention은 결국 '평균 내기'라서, 반복하면 모두 같은 값으로 수렴합니다 — over-smoothing)")
print("→ residual이 있으면 각 단어가 자기 입력을 계속 받아 구별이 유지되고,")
print("   그 위에 문맥 정보만 얹힙니다. 그래서 Transformer는 12층, 24층을 쌓을 수 있습니다.")

### G-6. Cross-Attention — Q는 디코더, K·V는 인코더

In [ ]:
src = ['the', 'black', 'cat', 'drank', 'milk']       # 영어 (인코더)
tgt = ['le', 'chat', 'noir', 'a', 'bu', 'du']         # 프랑스어 (디코더)
rng = np.random.default_rng(31)
X_enc = rng.normal(0, 1, (len(src), d_model))
X_dec = rng.normal(0, 1, (len(tgt), d_model))

def cross_attention(X_dec, X_enc, Wq, Wk, Wv):
    Q = X_dec @ Wq                 # Query  ← 디코더
    K = X_enc @ Wk                 # Key    ← 인코더
    Vv = X_enc @ Wv                # Value  ← 인코더
    A = softmax_rows(Q @ K.T / np.sqrt(Wq.shape[1]))
    return A @ Vv, A

out_x, A_x = cross_attention(X_dec, X_enc, Wq, Wk, Wv)
print("Cross-Attention 행렬 (행=프랑스어 출력, 열=영어 입력)")
print("          " + "".join(f"{w:>9}" for w in src))
for i, w in enumerate(tgt):
    print(f"{w:>9} " + "".join(f"{A_x[i,j]:9.3f}" for j in range(len(src))))
print(f"\n행렬 모양 {A_x.shape} — 정사각형이 아닙니다 (출력 6개 × 입력 5개).")
print("계산식은 self-attention과 완전히 동일하고, Q·K·V의 출처만 다릅니다.")

---
## H. 사전학습 — BERT 마스킹 · GPT 생성 · 프롬프트

### H-1. BERT의 15% / 80-10-10 마스킹 규칙 구현

In [ ]:
def bert_mask(tokens, vocab, p=0.15, mask_ratio=0.8, rand_ratio=0.1, seed=None):
    rng = np.random.default_rng(seed)
    out, labels = [], []
    for t in tokens:
        if rng.random() >= p:
            out.append(t); labels.append(None); continue
        r = rng.random()
        if r < mask_ratio:
            out.append('[MASK]')
        elif r < mask_ratio + rand_ratio:
            out.append(str(rng.choice(vocab)))
        else:
            out.append(t)
        labels.append(t)               # 예측해야 할 원래 단어
    return out, labels

sent2 = "어제 카페 에 갔었어 거기 사람 이 정말 많더라 그래서 금방 나왔어".split()
vocab2 = ['영화','학교','밥','파랑','달린다','컴퓨터','바다','노래']
for trial in range(3):
    masked, labels = bert_mask(sent2, vocab2, seed=trial)
    n_pred = sum(1 for l in labels if l is not None)
    print(f"[시도 {trial+1}] 예측 대상 {n_pred}/{len(sent2)}개")
    print("   입력:", ' '.join(masked))
    print("   정답:", ' '.join(l if l else '_' for l in labels))
    print()

In [ ]:
# 규칙대로 뽑히는지 통계로 검증 (10000회)
from collections import Counter
cnt = Counter()
rng = np.random.default_rng(0)
N = 10000
for _ in range(N):
    masked, labels = bert_mask(['w'], vocab2, seed=int(rng.integers(1e9)))
    if labels[0] is None:
        cnt['선택 안 됨'] += 1
    elif masked[0] == '[MASK]':
        cnt['[MASK] 치환'] += 1
    elif masked[0] == 'w':
        cnt['그대로 유지'] += 1
    else:
        cnt['랜덤 치환'] += 1

print(f"{'구분':>14} {'횟수':>8} {'비율':>9} {'기대값':>9}")
expect = {'선택 안 됨': 85.0, '[MASK] 치환': 12.0, '랜덤 치환': 1.5, '그대로 유지': 1.5}
for k in ['선택 안 됨', '[MASK] 치환', '랜덤 치환', '그대로 유지']:
    print(f"{k:>14} {cnt[k]:>8} {100*cnt[k]/N:>8.2f}% {expect[k]:>8.1f}%")

> 전체의 15% 중 80/10/10 이므로, 전체 기준으로는 **12% / 1.5% / 1.5%** 가 됩니다.
>
> **핵심질문 13 확인**: 만약 15% 전부를 `[MASK]` 로 바꾸면, 모델은 `[MASK]`가 있을 때만
> 문맥을 보도록 학습됩니다. 그런데 파인튜닝·추론 때는 `[MASK]`가 없으므로 그 능력이 발휘되지 않습니다.
> 랜덤 치환 10%는 "눈에 보이는 단어도 못 믿게" 만들어 항상 문맥을 확인하게 하고,
> 그대로 두기 10%는 `[MASK]` 없이도 좋은 표현을 만들게 합니다.

### H-2. GPT 스타일 자기회귀 생성 — Greedy / Temperature / Top-k

In [ ]:
# 아주 작은 문자 단위 모델을 흉내내기 위해, 다음 단어 확률표를 직접 정의
lm = {
    ('<s>',):        {'오늘': .5, '내일': .3, '어제': .2},
    ('오늘',):       {'날씨': .5, '기분': .3, '아침': .2},
    ('내일',):       {'날씨': .4, '시험': .4, '약속': .2},
    ('어제',):       {'카페': .6, '영화': .4},
    ('날씨',):       {'가': .7, '는': .3},
    ('기분',):       {'이': .8, '은': .2},
    ('카페',):       {'에서': .7, '에': .3},
}
default = {'좋다': .5, '그렇다': .3, '<eos>': .2}

def sample_next(hist, temperature=1.0, top_k=None, rng=None, greedy=False):
    dist = lm.get((hist[-1],), default)
    words = list(dist.keys())
    probs = np.array([dist[w] for w in words], dtype=float)
    if greedy:
        return words[int(np.argmax(probs))]
    logits = np.log(probs + 1e-12) / max(temperature, 1e-6)
    p = np.exp(logits - logits.max()); p /= p.sum()
    if top_k:
        idx = np.argsort(-p)[:top_k]
        mask = np.zeros_like(p); mask[idx] = 1
        p = p * mask; p /= p.sum()
    return str(rng.choice(words, p=p))

def generate(strategy, n=6, seed=0, **kw):
    rng = np.random.default_rng(seed)
    hist = ['<s>']
    for _ in range(n):
        w = sample_next(hist, rng=rng, **kw)
        if w == '<eos>':
            break
        hist.append(w)
    return ' '.join(hist[1:])

print("Greedy (항상 최고 확률):")
for s in range(3):
    print("  ", generate('greedy', greedy=True, seed=s))
print("\nTemperature = 1.0 (샘플링):")
for s in range(3):
    print("  ", generate('sample', temperature=1.0, seed=s))
print("\nTemperature = 0.3 (보수적):")
for s in range(3):
    print("  ", generate('sample', temperature=0.3, seed=s))
print("\nTemperature = 2.0 (창의적/불안정):")
for s in range(3):
    print("  ", generate('sample', temperature=2.0, seed=s))

> Greedy는 seed와 무관하게 **항상 같은 문장**을 냅니다 — 결정적이지만 단조롭습니다.
> Temperature를 올리면 다양해지지만 어색해질 위험이 커집니다.

### H-3. In-Context Learning 프롬프트 조립

In [ ]:
def build_prompt(task_desc, examples, question, cot=False, zero_shot_cot=False):
    parts = [task_desc]
    for q, a in examples:
        parts.append(f"{q}\n{a}")
    parts.append(question if not zero_shot_cot else question + "\nA: Let's think step by step.")
    return "\n\n".join(parts)

desc = "다음 산수 문장제를 풀어라."
q = "Q: The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"
ex_plain = [("Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
             "Each can has 3 tennis balls. How many tennis balls does he have now?",
             "A: The answer is 11.")]
ex_cot   = [("Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
             "Each can has 3 tennis balls. How many tennis balls does he have now?",
             "A: Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. "
             "5 + 6 = 11. The answer is 11.")]

configs = [
    ("Zero-shot",      build_prompt(desc, [], q)),
    ("One-shot",       build_prompt(desc, ex_plain, q)),
    ("Few-shot CoT",   build_prompt(desc, ex_cot, q)),
    ("Zero-shot CoT",  build_prompt(desc, [], q, zero_shot_cot=True)),
]
for name, p in configs:
    print("=" * 78)
    print(f"[{name}]  — 프롬프트 길이 {len(p)}자")
    print("-" * 78)
    print(p)
    print()

**핵심질문 16 확인**: 위 네 프롬프트는 모두 **모델의 가중치를 전혀 바꾸지 않습니다.**
바뀐 것은 오직 **모델에 넣는 텍스트**뿐인데, 논문 결과로는 MultiArith 정확도가 17.7 → 78.7 로 바뀝니다.
즉 능력이 새로 생긴 것이 아니라, 원래 있던 능력을 **꺼내 쓰도록 유도(elicit)** 한 것입니다.

---
## 마무리 체크리스트

아래 항목을 코드로 직접 확인했습니다. 하나씩 스스로 설명할 수 있는지 점검해 보세요.

- [ ] 원-핫 벡터는 왜 항상 직교하는가 (A-1)
- [ ] 차원이 커지면 왜 모든 벡터가 "거의 직교"가 되는가 (A-3)
- [ ] Skip-gram이 CBOW보다 학습쌍이 많은 이유 (B-1)
- [ ] Word2Vec의 목적은 예측이 아니라 벡터라는 것 (B-2)
- [ ] RNN이 가중치를 공유해야 가변 길이를 처리할 수 있는 이유 (C-2)
- [ ] $|W|<1$ 이면 기울기가 지수적으로 사라진다는 것 (C-3)
- [ ] forget gate 값이 기억의 반감기를 결정한다는 것 (D-1)
- [ ] LSTM의 cell state가 왜 "덧셈 지름길"인가 (D-2)
- [ ] $n$을 키우면 N-gram이 왜 무너지는가 (E-2)
- [ ] Greedy와 Beam Search의 결과가 왜 달라지는가 (E-3)
- [ ] 차원이 크면 왜 $\sqrt{d}$ 로 나눠야 하는가 (F-2)
- [ ] Masking에서 $-\infty$ 를 쓰는 이유 (G-2)
- [ ] head 수와 head당 차원의 트레이드오프 (G-3)
- [ ] Positional Encoding이 없으면 self-attention이 순서를 모른다는 것 (G-4)
- [ ] residual이 없으면 깊은 층에서 정보가 사라진다는 것 (G-5)
- [ ] Cross-Attention은 Q·K·V의 출처만 다르다는 것 (G-6)
- [ ] BERT의 80/10/10이 pretrain–finetune 불일치를 줄이기 위한 장치라는 것 (H-1)
- [ ] In-Context Learning은 가중치를 바꾸지 않는다는 것 (H-3)